# Recurrent Dynamic Range Extension

This Colab Notebook helps you running our HDR reconstruction method,<br>
introduced in our SIGGRAPH Asia 2026 paper \"Recurrent Dynamic Range Extension\".

Instructions:

1. Ensure using a GPU by setting \"Runtime/change runtime type\" to GPU.
2. Install the repository.
3. Read the example image or **put your own LDR images into './images/input'**.
4. Run our HDR reconstruction pipeline.
5. [optional] Download the result as EXR file.


In [ ]:
# Install required packages
!pip install colour-science rawpy lpips pytorch_lightning

In [ ]:
# Clone the repository.
!git clone https://github.com/sebastian-dille/RecurrentHDR
%cd RecurrentHDR

In [ ]:
import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
from src.utils import load_ldr
from src.extender import load_extension_model
from src.utils import tonemap
from inference import recurrent_inference 

We assume the image to be in linear RGB space.

The following function reads the LDR image and applies the EOFT BT.1886 as simple linearization.<br>
Of course, this can be replaced with a more accurate CRF correction if available.

In [ ]:
# Load the image to run through the pipeline.
img = load_ldr('./images/input/city.jpg')

First, let's run the extension model once. The input image is extended by a single exposure value:

In [ ]:
weights_url = 'https://github.com/sebastian-dille/RecurrentHDR/releases/download/v1.0/model_weights.pth'

# load the reconstruction models
extender = load_extension_model(weights_url,DEVICE)

# run the pipeline once - the image is extended by 1EV 
reconstructed_results = recurrent_inference(extender, img, steps=1, inference_size=1024)

# get the result
xdr = reconstructed_results['hdr']

Apply Reinhard's tone mapper to display the image:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.figure(figsize=(16,9))
plt.imshow(tonemap(xdr))
plt.axis('off')

In [ ]:
# OPTIONAL: run extender again on the extended output.
# This results in a 2EV increase overall.

# reconstructed_results = recurrent_inference(extender, xdr, steps=1)

Now, reconstruct the HDR highlights with our automatic recurrent pipeline:

In [ ]:
# run the pipeline
reconstructed_results = recurrent_inference(extender, img, inference_size=1024)

# get the result
hdr = reconstructed_results['hdr']

In [ ]:
import numpy as np
plt.figure(figsize=(16,9))
plt.imshow(tonemap(hdr))
plt.axis('off')